# T15 — DDL Import: SQL to Synthetic Data

Import SQL DDL (CREATE TABLE statements) and generate synthetic data from any existing database design.

**What you'll learn:**
- Parse SQL DDL into a Spindle schema
- Understand type-to-strategy mapping and column name heuristics
- Generate data from the inferred schema
- Verify FK integrity across generated tables

## 1. Define SQL DDL

In [1]:
ddl = """
CREATE TABLE customer (
    customer_id INT IDENTITY(1,1) NOT NULL,
    first_name NVARCHAR(50) NOT NULL,
    last_name NVARCHAR(50) NOT NULL,
    email NVARCHAR(100),
    loyalty_tier VARCHAR(20),
    created_at DATETIME2,
    CONSTRAINT PK_customer PRIMARY KEY (customer_id)
);

CREATE TABLE product (
    product_id INT IDENTITY(1,1) NOT NULL,
    product_name NVARCHAR(100) NOT NULL,
    category VARCHAR(50),
    unit_price DECIMAL(10,2),
    is_active BIT DEFAULT 1,
    CONSTRAINT PK_product PRIMARY KEY (product_id)
);

CREATE TABLE [order] (
    order_id INT IDENTITY(1,1) NOT NULL,
    customer_id INT NOT NULL,
    order_date DATE NOT NULL,
    total DECIMAL(10,2),
    status VARCHAR(20),
    CONSTRAINT PK_order PRIMARY KEY (order_id),
    CONSTRAINT FK_order_customer FOREIGN KEY (customer_id)
        REFERENCES customer(customer_id)
);

CREATE TABLE order_line (
    line_id INT IDENTITY(1,1) NOT NULL,
    order_id INT NOT NULL,
    product_id INT NOT NULL,
    quantity INT,
    line_total DECIMAL(10,2),
    CONSTRAINT PK_order_line PRIMARY KEY (line_id),
    CONSTRAINT FK_line_order FOREIGN KEY (order_id) REFERENCES [order](order_id),
    CONSTRAINT FK_line_product FOREIGN KEY (product_id) REFERENCES product(product_id)
);
"""
print(f"DDL defines 4 tables with explicit FK constraints")

DDL defines 4 tables with explicit FK constraints


## 2. Parse DDL into Spindle Schema

In [2]:
from sqllocks_spindle.schema.ddl_parser import DdlParser

parser = DdlParser()
schema = parser.parse_string(ddl)

print(f"Tables: {list(schema.tables.keys())}")
print(f"Relationships: {len(schema.relationships)}")
for r in schema.relationships:
    print(f"  {r.child}.{r.child_columns} -> {r.parent}.{r.parent_columns}")

Tables: ['customer', 'product', 'order', 'order_line']
Relationships: 3
  order.['customer_id'] -> customer.['customer_id']
  order_line.['order_id'] -> order.['order_id']
  order_line.['product_id'] -> product.['product_id']


## 3. Inspect Inferred Strategies

In [3]:
for table_name, table_def in schema.tables.items():
    print(f"\n=== {table_name} ===")
    for col_name, col_def in table_def.columns.items():
        strategy = col_def.generator.get('strategy', 'unknown')
        print(f"  {col_name:20s} -> {strategy}")


=== customer ===
  customer_id          -> sequence
  first_name           -> faker
  last_name            -> faker
  email                -> faker
  loyalty_tier         -> faker
  created_at           -> temporal

=== product ===
  product_id           -> sequence
  product_name         -> faker
  category             -> faker
  unit_price           -> distribution
  is_active            -> weighted_enum

=== order ===
  order_id             -> sequence
  customer_id          -> foreign_key
  order_date           -> temporal
  total                -> distribution
  status               -> weighted_enum

=== order_line ===
  line_id              -> sequence
  order_id             -> foreign_key
  product_id           -> foreign_key
  quantity             -> distribution
  line_total           -> distribution


## 4. Generate Synthetic Data

In [4]:
from sqllocks_spindle import Spindle

spindle = Spindle()
result = spindle.generate(schema=schema, scale="small", seed=42)
print(result.summary())

Spindle Generation Result
Schema: ddl_import
Domain: custom
Mode:   3nf
Seed:   42
Time:   0.4s

Table                             Rows  Columns
---------------------------------------------
customer                         1,000        6
order                            2,500        5
product                          1,000        5
order_line                       2,500        5
---------------------------------------------
TOTAL                            7,000


## 5. Verify FK Integrity

In [5]:
errors = result.verify_integrity()
if errors:
    for e in errors:
        print(f"ERROR: {e}")
else:
    print("All FK relationships are valid!")

# Inspect generated data
print(f"\nCustomers: {len(result['customer'])} rows")
result['customer'].head()

All FK relationships are valid!

Customers: 1000 rows


,customer_id,first_name,last_name,email,loyalty_tier,created_at
0,1,Dillon,Macdonald,hyoung@example.org,Amount follow sort p,2025-09-18 18:38:08.004650719
1,2,Andrew,Paul,brianna69@example.com,Score evidence whose,2025-05-24 01:53:16.328832200
2,3,Jasmine,Schaefer,lsmith@example.net,Thing wall material,2024-03-09 17:59:13.685969832
3,4,John,Evans,walvarez@example.net,Without term meeting,2025-12-13 04:54:12.962433472
4,5,Andrea,Ramos,teresa60@example.org,Lead condition past,2025-07-09 15:10:03.283935546


## 6. CLI Equivalent

```bash
# Save DDL to file, then:
spindle from-ddl my_tables.sql --output my_schema.spindle.json
spindle generate --schema my_schema.spindle.json --scale small --output ./data/
```